# Lets do cosior analysis on the data expressions!

In [1]:
from circadian_cell_painting import (
    detect,
    CosinorDetection,
    CosinorVisualize,
    OscillationResults,
    cosinor_fitted_long
)
from pathlib import Path
import polars as pl
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import string
from tqdm import tqdm
import numpy as np
from scipy.signal import savgol_filter
import pandas as pd
from natsort import natsorted
import matplotlib.colors as mcolors

dataset = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/"
               "enVision/enVision_preprocessed/2026-04-01._.PREPROCESSED."
               "parquet")


lf = pl.scan_parquet(dataset).filter(pl.col("feature") == "CalcResultI")

tpoints = lf.select("Timepoint").unique().sort(by="Timepoint").collect().to_series().to_list()

platemap = (
    pl.read_excel(
    "/home/derfelt/PartchLabFiles/Circadian"
    " Rhythms/Live Study/2026-04-01Echo_transfer.xlsx", engine="openpyxl")
    .select(["384_Well", 
             "MoleculeID", 
             "strain",
             "dex"])
    .unique(subset=["384_Well"])
    .with_columns(
        pl.col("MoleculeID").str.strip_chars()
    )
    ).rename({"384_Well": "Well"})

display(lf.head(2).collect())
display(platemap)

# print(*tpoints,sep="\n")

Well,Timepoint,feature,mean,median,stddev,q0,q10,q25,q50,q75,q90,q100
str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A01""",24.0,"""CalcResultI""",589.67097,589.67097,null,589.67097,589.67097,589.67097,589.67097,589.67097,589.67097,589.67097
"""A01""",24.5,"""CalcResultI""",425.256143,425.256143,null,425.256143,425.256143,425.256143,425.256143,425.256143,425.256143,425.256143


Well,MoleculeID,strain,dex
str,str,str,str
"""F15""","""MH-23 100uM""","""Bmal1""","""dex"""
"""F05""","""noDMSO""","""Bmal1""","""NoSynhronisation"""
"""K14""","""MK-1804 25uM""","""Per2""","""dex"""
"""C10""","""MH-13 25uM""","""Bmal1""","""dex"""
"""N12""","""MK-0769 50uM""","""Per2""","""dex"""
…,…,…,…
"""C11""","""Etoposide 1uM""","""Bmal1""","""dex"""
"""E07""","""MH-20 50uM""","""Bmal1""","""dex"""
"""M08""","""LY 5uM""","""Per2""","""dex"""


In [ ]:
out_file = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/cosinor_anal"
                "ysis/2026-04._.RESULTS.parquet")

out_pickle = Path("/home/derfelt/PartchLabFiles/Circadian"
                  " Rhythms/Live Study/enVision/enVision_pickle")
out_pickle = out_pickle / "2026-04._.obj.pickle"

model = CosinorDetection(intervals=1)

result: OscillationResults = detect(
    file_path=dataset,
    platemap_path=platemap,
    feature_col="feature",
    well_column="Well",
    timepoint_col="Timepoint",
    value_col="mean",
    stat="mean",
    out_file=out_file,
    correction_group=["MoleculeID", "strain"],
    detector=model
)

result.to_pickle(out_pickle)


In [2]:
out_pickle = Path("/home/derfelt/PartchLabFiles/Circadian"
                  " Rhythms/Live Study/enVision/enVision_pickle")

out_pickle = out_pickle / "2026-04._.obj.pickle"

result = OscillationResults.load_pickle(out_pickle)

display(result.scan_results().collect())

Well,feature,stat,baseline,amplitude,phase,p_value,f_stat,ss_model,ss_null,fitted,MoleculeID,strain,dex,p_adjusted
str,str,str,f64,f64,f64,f64,f64,f64,f64,list[f64],str,str,str,f64
"""A01""","""Signal""","""mean""",-5.725383,34.750229,0.840204,1.2738e-7,17.882455,326544.303987,412418.03523,"[17.463836, 20.643629, … -3.083638]",null,null,null,1.6336e-7
"""A01""","""CalcResultI""","""mean""",-28.626915,173.751145,0.840204,1.2738e-7,17.882455,8.1636e6,1.0310e7,"[87.31918, 103.218143, … -15.418195]",null,null,null,1.6336e-7
"""A02""","""Signal""","""mean""",-7.009078,38.64622,0.760112,2.2721e-7,17.15466,419172.051452,524918.435082,"[21.000111, 24.236055, … -0.997596]",null,null,null,2.7644e-7
"""A02""","""CalcResultI""","""mean""",-35.045393,193.231099,0.760112,2.2721e-7,17.15466,1.0479e7,1.3123e7,"[105.000563, 121.18028, … -4.98797]",null,null,null,2.7644e-7
"""A03""","""CalcResultI""","""mean""",-29.095878,186.537792,0.758359,3.7988e-8,19.424229,8.6239e6,1.1087e7,"[106.32416, 121.910542, … 0.243394]",null,null,null,5.4375e-8
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P22""","""Signal""","""mean""",4.066448,248.722938,0.244446,3.6514e-12,32.163157,8.9459e6,1.3177e7,"[-237.262363, -243.054858, … -150.747838]",null,null,null,1.0880e-11
"""P23""","""CalcResultI""","""mean""",-8.632235,1408.794446,0.278629,5.5611e-12,31.54541,2.9328e8,4.2933e8,"[-1363.09433, -1402.081909, … -847.321512]",null,null,null,1.4499e-11
"""P23""","""Signal""","""mean""",-1.726451,281.75889,0.278629,5.5611e-12,31.54541,1.1731e7,1.7173e7,"[-272.61887, -280.416386, … -169.464304]",null,null,null,1.4499e-11


# let's visualize the results

In [ ]:
result_lf = pl.scan_parquet(result.results_path).filter(pl.col("feature") == "CalcResultI")
dataset_lf = pl.scan_parquet(dataset).filter(pl.col("feature") == "CalcResultI").select(["Well", "Timepoint", "feature", "mean"])
dataset_lf = dataset_lf.join(other=platemap.lazy(), on="Well", how="left").sort(["Well", "Timepoint"])

# smoothing option
dataset_lf = (dataset_lf
                .collect()
                .group_by("Well")
                .map_groups(
                    lambda g: g.with_columns(
                      pl.Series("mean", savgol_filter(
                          g["mean"].to_numpy(),
                          window_length=13,
                          polyorder=3
                      ))
                    )
                )
              )

#grab only dex
dataset_lf = (dataset_lf
              .filter(pl.col('dex') == 'dex')
              .sort(["dex","strain","MoleculeID", "Well", "Timepoint"], descending=[True, True, True, False, False]))

display(dataset_lf)

# result_lf.drop('fitted').sink_csv("/home/derfelt/PartchLabFiles/Circadian "
#                 "Rhythms/Live Study/enVision/cosinor_anal"
#                 "ysis/2026-04._.RESULTS.csv")

# display(dataset_lf.head(10).collect())
# display(result_lf.head(10).collect().select())

# print(result_lf.collect().shape)

In [ ]:
# collect and join
import pandas as pd
from scipy.signal import savgol_filter

dataset_pddf = dataset_lf.to_pandas()
wells_needed = dataset_pddf["Well"].unique()
result_pddf = result_lf.collect().to_pandas()

timepoints = sorted(dataset_pddf["Timepoint"].unique())

# display(dataset_pddf, result_pddf)


def expand_fitted(result_pddf, tpoints):
    rows = []
    for _, row in result_pddf.iterrows():
        fitted_vals = row["fitted"]          # list of floats
        if fitted_vals is None or len(fitted_vals) != len(tpoints):
            continue
        for t, f in zip(tpoints, fitted_vals):
            rows.append({"Well": row["Well"], "Timepoint": t, "fitted": f,
                         "MoleculeID": row["MoleculeID"], "strain": row["strain"],
                         "dex": row["dex"], "baseline": row["baseline"],
                         "amplitude": row["amplitude"], "phase": row["phase"],
                         "p_value": row["p_value"], "p_adjusted": row["p_adjusted"]})
    return pd.DataFrame(rows)

fitted_long = expand_fitted(result_pddf, timepoints)

display(fitted_long, dataset_pddf)

# fitted_long.to_csv("/home/derfelt/PartchLabFiles/Circadian "
#                 "Rhythms/Live Study/enVision/cosinor_anal"
#                 "ysis/2026-04._.RESULTS-LONG_CSV.csv")

## lets visualize each curve and see how it looks like

In [3]:
def make_title(row: dict, well_col: str, meta_cols: list[str]) -> str:
    parts = [row[well_col]]
    for col in meta_cols:
        val = row.get(col)
        if val is not None and str(val).strip() != '' and str(val).lower() != 'nan':
            parts.append(str(val))
    return ' | '.join(parts)

def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max + interval, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

def get_shades(base_color: str, n: int) -> list:
    """light → dark shades of base_color"""
    rgb = mcolors.to_rgb(base_color)
    return [tuple(1 - (1 - c) * (i + 1) / n for c in rgb) for i in range(n)]

def get_grays(n: int) -> list:
    return [str(round(0.3 + 0.6 * i / max(n - 1, 1), 2)) for i in range(n)]

### Load and format data

In [13]:
detrended_data = result.scan_data()
cosinor_data = result.scan_results().filter(pl.col("feature") == "CalcResultI")

well_columns = "Well"
wells = cosinor_data.select(well_columns).unique().sort(well_columns).collect()[well_columns].to_list()

tpoint_col = "Timepoint"
stat_values = "mean"

# smoothing detrended data
win_len = 13
polyorder = 3

meta_columns = ["MoleculeID", "strain", "dex"]

schema = detrended_data.schema
detrended_data = (detrended_data
                .filter(pl.col("feature") == "CalcResultI")
                .select([well_columns, tpoint_col, "feature", stat_values])
                .group_by("Well")
                .map_groups(
                    lambda g: 
                        g.with_columns(
                            pl.Series(
                                stat_values,
                                savgol_filter(
                                    g[stat_values].to_numpy(),
                                    window_length=win_len,
                                    polyorder=polyorder
                                )
                            )
                        ),
                    schema=schema
                ).sort([well_columns, tpoint_col])
                .join(platemap.lazy(), on='Well', how='left')
                .sort(["dex","strain","MoleculeID", "Well", "Timepoint"], descending=[True, True, True, False, False])
            ).collect().to_pandas()

tpoints = [float(t) for t in sorted(detrended_data[tpoint_col].unique())]

cosinor_long = cosinor_fitted_long(cosinor_data, tpoints, tpoint_col).to_pandas()

cosinor_data = cosinor_data.collect().to_pandas()

detrended_data['group_label'] = detrended_data[meta_columns].apply(
    lambda row: ' | '.join([str(v) for v in row]), axis=1
)
cosinor_long['group_label'] = cosinor_long[meta_columns].apply(
    lambda row: ' | '.join([str(v) for v in row]), axis=1
)

display(detrended_data, cosinor_long)
# display(detrended_data.head(100).collect(), cosinor_data.head(100).collect())


# display(wells)

/tmp/ipykernel_27257/580725465.py:16: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = detrended_data.schema


,Well,Timepoint,feature,mean,MoleculeID,strain,dex,group_label
0,A01,24.0,CalcResultI,573.864667,NaN,NaN,NaN,nan | nan | nan
1,A01,24.5,CalcResultI,581.099542,NaN,NaN,NaN,nan | nan | nan
2,A01,25.0,CalcResultI,569.526612,NaN,NaN,NaN,nan | nan | nan
3,A01,25.5,CalcResultI,541.428043,NaN,NaN,NaN,nan | nan | nan
4,A01,26.0,CalcResultI,499.085997,NaN,NaN,NaN,nan | nan | nan
...,...,...,...,...,...,...,...,...
53371,E12,91.5,CalcResultI,-143.273064,0.01% DMSO,Bmal1,NoSynhronisation,0.01% DMSO | Bmal1 | NoSynhronisation
53372,E12,92.0,CalcResultI,-153.361100,0.01% DMSO,Bmal1,NoSynhronisation,0.01% DMSO | Bmal1 | NoSynhronisation
53373,E12,92.5,CalcResultI,-163.176103,0.01% DMSO,Bmal1,NoSynhronisation,0.01% DMSO | Bmal1 | NoSynhronisation
53374,E12,93.0,CalcResultI,-173.575571,0.01% DMSO,Bmal1,NoSynhronisation,0.01% DMSO | Bmal1 | NoSynhronisation


,Well,feature,stat,baseline,amplitude,phase,p_value,f_stat,ss_model,ss_null,fitted,MoleculeID,strain,dex,p_adjusted,Timepoint,group_label
0,A01,CalcResultI,mean,-28.626915,173.751145,0.840204,1.273807e-07,17.882455,8.163608e+06,1.031045e+07,87.319180,NaN,NaN,NaN,1.633643e-07,24.0,nan | nan | nan
1,A01,CalcResultI,mean,-28.626915,173.751145,0.840204,1.273807e-07,17.882455,8.163608e+06,1.031045e+07,103.218143,NaN,NaN,NaN,1.633643e-07,24.5,nan | nan | nan
2,A01,CalcResultI,mean,-28.626915,173.751145,0.840204,1.273807e-07,17.882455,8.163608e+06,1.031045e+07,116.861200,NaN,NaN,NaN,1.633643e-07,25.0,nan | nan | nan
3,A01,CalcResultI,mean,-28.626915,173.751145,0.840204,1.273807e-07,17.882455,8.163608e+06,1.031045e+07,128.014916,NaN,NaN,NaN,1.633643e-07,25.5,nan | nan | nan
4,A01,CalcResultI,mean,-28.626915,173.751145,0.840204,1.273807e-07,17.882455,8.163608e+06,1.031045e+07,136.488447,NaN,NaN,NaN,1.633643e-07,26.0,nan | nan | nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53371,P24,CalcResultI,mean,78.635074,1406.971605,0.346333,5.440315e-12,31.577550,2.935598e+08,4.298819e+08,13.422519,NaN,NaN,NaN,1.449860e-11,91.5,nan | nan | nan
53372,P24,CalcResultI,mean,78.635074,1406.971605,0.346333,5.440315e-12,31.577550,2.935598e+08,4.298819e+08,-169.468856,NaN,NaN,NaN,1.449860e-11,92.0,nan | nan | nan
53373,P24,CalcResultI,mean,78.635074,1406.971605,0.346333,5.440315e-12,31.577550,2.935598e+08,4.298819e+08,-348.115104,NaN,NaN,NaN,1.449860e-11,92.5,nan | nan | nan
53374,P24,CalcResultI,mean,78.635074,1406.971605,0.346333,5.440315e-12,31.577550,2.935598e+08,4.298819e+08,-519.459538,NaN,NaN,NaN,1.449860e-11,93.0,nan | nan | nan


stats to report:

- baseline	
- amplitude	
- phase
- p_adjusted

In [8]:
import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
from tqdm import tqdm

out_file = Path("/home/derfelt/PartchLabFiles/Circadian "
                "Rhythms/Live Study/enVision/enVision_vis"
                "ual_results/cosinor_fitted_per_well.pdf")


with PdfPages(out_file) as pdf:
    for well in tqdm(wells, desc="Well procesing"):
        
        well_detrended_df = (detrended_data[detrended_data[well_columns] == well]).sort_values(tpoint_col)
        well_cosinor = cosinor_data[cosinor_data[well_columns] == well]
        
        if well_cosinor.empty or well_detrended_df.empty:
            continue
        
        timepoints = sorted(well_detrended_df[tpoint_col].values)
        # print(timepoints)
        
        fitted = well_cosinor['fitted'].to_numpy()[0]
        # print(fitted)
        
        # display(well_detrended_df, well_cosinor)
        
        fig, ax = plt.subplots(figsize=(12,8))
        
        sns.lineplot(data=well_detrended_df, x=tpoint_col, y=stat_values, ax=ax, color='royalblue', linewidth=2, label='detrended data')
        ax.plot(timepoints, fitted, color='crimson', linewidth=2, linestyle='--', label='cosinor fit', zorder=10)
        
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        
        add_period_marks(ax, min(timepoints), max(timepoints))
        
        mol = str(well_cosinor['MoleculeID'].values[0])
        strain = str(well_cosinor['strain'].values[0])
        
        row = well_cosinor.iloc[0].to_dict()
        
        # ax.set_title(f"{'Well:':<15} {well} - {mol} | {strain}\n"
        #              f"{'Baseline:':<15} {row['baseline']:.4f}\n"
        #              f"{'Amplitude:':<15} {row['amplitude']:.4f}\n"
        #              f"{'Phase:':<15} {row['phase']:.4f}",
        #              loc='center',
        #              fontsize=14,
        #              fontfamily='monospace'
        #              )
        ax.set_xlabel('Time (hours)')
        ax.set_ylabel('Expression')
        ax.legend(fontsize=8)
        
        fig.suptitle(
            f"Well: {well} - {mol} | {strain}",
            fontsize=12,
            fontfamily='monospace',
        )
        stats_text = (
            f"Baseline: {row['baseline']:.4f}    "
            f"Amplitude: {row['amplitude']:.4f}    "
            f"Phase: {row['phase']:.4f}    "
            f"p_adj: {row['p_adjusted']:.4e}    "
        )

        ax.set_title(stats_text, fontfamily='monospace', fontsize=10)

        # plt.subplots_adjust(top=0.88)  # make room for the two title lines
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

Well procesing: 100%|██████████| 384/384 [00:49<00:00,  7.75it/s]


### Plotting grouped concentrations

In [9]:
PLOT_CONFIGS = [
    {
        'title': 'DMSO Controls',
        'drugs': [],
        'dmso' : ['0.005% DMSO', '0.01% DMSO', '0.05% DMSO',
                  '0.1% DMSO', '0.25% DMSO', '0.5% DMSO', '1% DMSO'],
    },
    {
        'title': 'MH-11 & MH-23',
        'drugs': [('MH-11', 'steelblue'), ('MH-23', 'darkorange')],
        'dmso' : ['0.05% DMSO'],
    },
    {
        'title': 'MH-13 & MH-20',
        'drugs': [('MH-13', 'steelblue'), ('MH-20', 'darkorange')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title': 'PF & Etoposide',
        'drugs': [('PF', 'steelblue'), ('Etoposide', 'darkorange')],
        'dmso' : ['0.01% DMSO'],
    },
    {
        'title': 'LY, SR, Nobiletin, Naringine, T0901317',
        'drugs': [
            ('LY', 'steelblue'), ('SR', 'darkorange'),
            ('Nob', 'green'), ('Nar', 'purple'), ('T0901317', 'brown')
        ],
        'dmso' : ['0.05% DMSO', '0.01% DMSO'],
    },
    {
        'title': 'Bacterial & Aswad',
        'drugs': [
            ('E.coli#36', 'steelblue'), ('S.pneumo#6', 'darkorange'),
            ('Aswad A0005', 'green'), ('Aswad A0010', 'purple'), ('Aswad A0024', 'brown')
        ],
        'dmso' : ['0.05% DMSO'],
    },
    {
        'title': 'KL001',
        'drugs': [('KL001', 'steelblue')],
        'dmso' : ['0.01% DMSO'],
    },
    {
        'title': 'MK Compounds',
        'drugs': [
            ('MK-0761', 'steelblue'), ('MK-0769', 'darkorange'),
            ('MK-1282', 'green'),     ('MK-1496', 'purple'),
            ('MK-1804', 'brown'),     ('MK-2267', 'red'),
            ('MK-8692', 'teal'),      ('MK-8800', 'olive'),
        ],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-0761',
        'drugs' : [('MK-0761', 'steelblue')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-1282',
        'drugs' : [('MK-1282', 'green')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-1804',
        'drugs' : [('MK-1804', 'brown')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-8692',
        'drugs' : [('MK-8692', 'teal')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-0769',
        'drugs' : [('MK-0769', 'darkorange')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-1496',
        'drugs' : [('MK-1496', 'purple')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-2267',
        'drugs' : [('MK-2267', 'red')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
    {
        'title' : 'MK-8800',
        'drugs' : [('MK-8800', 'olive')],
        'dmso' : ['1% DMSO', '0.5% DMSO', '0.25% DMSO'],
    },
]

In [10]:
well_columns = "Well"

tpoint_col = "Timepoint"
stat_values = "mean"

strains = cosinor_data['strain'].dropna().unique()

def plot_config_groups(ax, detrended_df, fitted_df, config, strain = None, colored_dmso = True):
    d_df = detrended_df.copy()
    f_df = fitted_df.copy()
    
    if strain:
        d_df = d_df[d_df['strain'] == strain]
        f_df = f_df[f_df['strain'] == strain]
    
    
    def _plot_group(mol_mask_d, mol_mask_f, color, label, ax, linewidth=1.5, zorder=5):
        sub_d = d_df[mol_mask_d]
        sub_f = f_df[mol_mask_f]
        # display(sub_d, sub_f)
        
        if sub_d.empty:
            return
        
        # detrended data
        sns.lineplot(
            data=sub_d, x=tpoint_col, y=stat_values,
            errorbar='sd', color=color, linewidth=linewidth,
            linestyle='-', label=f'{label} (raw)', ax=ax,
            zorder=zorder, err_kws={'alpha': 0.1, 'facecolor': color}
        )
        
        # fitted curve
        if not sub_f.empty:
            sns.lineplot(
                data=sub_f, x=tpoint_col, y='fitted',
                errorbar='sd', color=color, linewidth=linewidth,
                linestyle='--', label=f"{label} (fit)", ax=ax,
                zorder=zorder+1, err_kws={'alpha':0.1, 'facecolor':color}
            )
    
    
    dmso_list = config['dmso']
    
    if not config['drugs'] and colored_dmso:
        palette = sns.color_palette("tab10", len(dmso_list))
        for dmso, color in zip(dmso_list, palette):
            _plot_group(
                d_df['MoleculeID'] == dmso,
                f_df['MoleculeID'] == dmso,
                color=color, label=dmso, ax=ax, linewidth=2, zorder=10
            )
    else:
        for drug, base_color in config['drugs']:
            drug_d = d_df[d_df['MoleculeID'].str.startswith(drug, na=False)]
            drug_f = f_df[f_df['MoleculeID'].str.startswith(drug, na=False)]
            concentrations = natsorted(drug_d['MoleculeID'].unique().tolist())
            shades = get_shades(base_color, len(concentrations))
            
            for conc, shade in zip(concentrations, shades):
                _plot_group(
                    d_df['MoleculeID'] == conc,
                    f_df['MoleculeID'] == conc,
                    color=shade, label=conc, ax=ax
                )
        
        # DMSOs    
        grays = get_grays(len(dmso_list))
        for dmso, grays in zip(dmso_list, grays):
            _plot_group(
                d_df['MoleculeID'] == dmso,
                f_df['MoleculeID'] == conc,
                color='black', label=dmso, ax=ax, linewidth=2, zorder=10
            )
        
    
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    title = config['title']
    if strain:
        title += f' - {strain}'
        
    
    ax.set_title(title)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Expression')
    ax.legend(fontsize=6, loc='upper right', ncol=2)
    add_period_marks(ax, d_df[tpoint_col].min(), d_df[tpoint_col].max())


In [11]:
out_file = Path('/home/derfelt/PartchLabFiles/'
                'Circadian Rhythms/Live Study/'
                'enVision/enVision_visual_results')
out_file = out_file / "cosinor_grouped_analysis.pdf"

with PdfPages(out_file) as pdf:
    for config in tqdm(PLOT_CONFIGS, desc='plot configs'):
        fig, axs = plt.subplots(2, 1, figsize=(24,10), sharey=False)
        for ax, strain in zip(axs, strains):
            plot_config_groups(ax=ax, 
                               detrended_df=detrended_data,
                               fitted_df=fitted_long,
                               config=config,
                               strain=strain,
                               colored_dmso=True)
            
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

plot configs: 100%|██████████| 16/16 [00:20<00:00,  1.26s/it]


### Plotting individual replicates

In [14]:
out_file = Path("/home/derfelt/PartchLabFiles/Circadian"
                " Rhythms/Live Study/enVision/enVision_"
                "visual_results")
out_file = out_file / "cosinor_grouped_results.pdf"

with PdfPages(out_file) as pdf:
    groups = detrended_data['group_label'].unique()
    
    for group in tqdm(groups, 'plotting groups'):
        gdf_detrended = detrended_data[detrended_data['group_label'] == group]
        gdf_fitted = cosinor_long[cosinor_long['group_label'] == group]
        
        fig, ax = plt.subplots(figsize=(14,8))
        
        # detrended signal
        sns.lineplot(gdf_detrended, x=tpoint_col, y=stat_values,
                     errorbar='sd', color='royalblue', linewidth=1.5,
                     label="detrended data", ax=ax, err_kws={
                         'alpha': 0.2, 'facecolor': 'royalblue'
                     })
        
        # fitted curve
        if not gdf_fitted.empty:
            sns.lineplot(gdf_fitted,
                         x=tpoint_col, y='fitted',
                         errorbar='sd', color='crimson', linewidth=2,
                         linestyle='--', label='cosinor fitted curve', ax=ax,
                         err_kws={'alpha':0.2, 'facecolor':'crimson'})
        
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.set_title(group)
        ax.set_xlabel('Time (hours)')
        ax.set_ylabel('Expression')
        ax.legend(fontsize=8)
        
        add_period_marks(ax, gdf_detrended[tpoint_col].min(), gdf_detrended[tpoint_col].max())
        
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
        

plotting groups: 100%|██████████| 129/129 [00:14<00:00,  8.95it/s]
